# Notebook 3 — Choosing an Embedding Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikapv-kv/ai-upskill-layer-3/blob/main/notebooks/03_embedding_model_comparison.ipynb)

**Story so far:** semantic search works on the Acme knowledge base (NB2) using MiniLM. **Is MiniLM the right model for Acme?** Don't guess; measure.

**You'll compare:** `all-MiniLM-L6-v2` vs `BAAI/bge-small-en-v1.5` on size, speed and retrieval quality for Acme questions.

🎤 **In session:** the instructor shows the results · 🏠 **Take-home:** run it yourself + E3.1 · Runtime: CPU

## 0. Setup

In [ ]:
%pip install -q sentence-transformers==6.1.0

In [ ]:
import json, os, time, urllib.request
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # Apache-2.0
ALT_MODEL_NAME = "BAAI/bge-small-en-v1.5"               # MIT
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # from the BGE model card

models = {MODEL_NAME: SentenceTransformer(MODEL_NAME), ALT_MODEL_NAME: SentenceTransformer(ALT_MODEL_NAME)}
query_prefixes = {MODEL_NAME: "", ALT_MODEL_NAME: BGE_QUERY_PREFIX}

DATA_BASE_URL = "https://raw.githubusercontent.com/karthikapv-kv/ai-upskill-layer-3/main/data"

def data_file(relative_path):
    """Local path to a workshop data file; downloads it from GitHub if needed."""
    for folder in ["data", "../data"]:
        path = os.path.join(folder, relative_path)
        if os.path.exists(path):
            return path
    path = os.path.join("data", relative_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(f"{DATA_BASE_URL}/{relative_path}", path)
    return path

with open(data_file("kb_documents.json")) as f:
    documents = json.load(f)
with open(data_file("eval_queries.json")) as f:
    eval_queries = json.load(f)

doc_texts = [d["title"] + ". " + d["content"] for d in documents]
print(len(documents), "documents,", len(eval_queries), "evaluation queries")

## 1. Size and shape

In [ ]:
rows = []
for name, model in models.items():
    rows.append({
        "model": name,
        "dimensions": model.encode(["test"]).shape[1],
        "parameters (millions)": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
        "max input tokens": model.max_seq_length,
    })
pd.DataFrame(rows)

**Expected:** both **384** dimensions; MiniLM ≈ **22.7 M** parameters, BGE ≈ **33.4 M**; max input **256** vs **512** tokens.

## 2. Speed
Time to embed all 18 docs. Run once to warm up, then time it. Depends on your hardware.

In [ ]:
doc_embeddings = {}
for name, model in models.items():
    model.encode(doc_texts[:2])                        # warm-up
    start = time.perf_counter()
    doc_embeddings[name] = model.encode(doc_texts, normalize_embeddings=True)
    print(f"{name:<42} {(time.perf_counter() - start) * 1000:6.0f} ms")

**Expected:** tens of milliseconds each on a typical CPU; the exact numbers vary by machine. Both are fast enough for Acme's 18 docs; at millions of docs, the difference would matter.

## 3. Quality on Acme questions
`eval_queries.json`: 15 realistic questions, each with the document that should answer it.

In [ ]:
pd.DataFrame(eval_queries)

**hit@k** = how often the expected document is in the top *k* results.

In [ ]:
def evaluate_model(model, eval_queries, documents, k=3, query_prefix="", doc_vectors=None):
    """Rank of the expected document for each query, plus hit@1 and hit@k."""
    if doc_vectors is None:
        doc_vectors = model.encode([d["title"] + ". " + d["content"] for d in documents], normalize_embeddings=True)
    ids = [d["document_id"] for d in documents]
    ranks = []
    for item in eval_queries:
        query_vector = model.encode([query_prefix + item["query"]], normalize_embeddings=True)[0]
        order = [ids[i] for i in np.argsort(-(doc_vectors @ query_vector))]
        ranks.append(order.index(item["expected_document_id"]) + 1)
    return {"hit@1": sum(r == 1 for r in ranks), f"hit@{k}": sum(r <= k for r in ranks), "ranks": ranks}

results = {name: evaluate_model(model, eval_queries, documents, query_prefix=query_prefixes[name],
                                doc_vectors=doc_embeddings[name])
           for name, model in models.items()}

for name, r in results.items():
    print(f"{name:<42} hit@1 = {r['hit@1']}/{len(eval_queries)}   hit@3 = {r['hit@3']}/{len(eval_queries)}")

**Expected:** MiniLM **14/15** · **15/15**; BGE **12/15** · **14/15**.

### Where do they disagree?

In [ ]:
table = pd.DataFrame({
    "query": [q["query"] for q in eval_queries],
    "expected": [q["expected_document_id"] for q in eval_queries],
    "MiniLM rank": results[MODEL_NAME]["ranks"],
    "BGE rank": results[ALT_MODEL_NAME]["ranks"],
})
table[table["MiniLM rank"] != table["BGE rank"]]

**Expected:** 4 rows. Mostly MiniLM wins (e.g. *"One malformed event is blocking the whole partition"*, KB-008: MiniLM 1st, BGE 7th), but BGE wins one (the INC-2291 outage question, KB-017).

**Takeaway:** BGE is often regarded as the stronger retrieval model, but on **Acme's** questions MiniLM did slightly better. With 15 queries, 2 wins could be noise. **Test on your own queries**, and use more than 15.

## 4. Does BGE's query prefix matter here?
The BGE model card recommends a prefix on *queries* (not documents).

In [ ]:
bge = models[ALT_MODEL_NAME]
for prefix, label in [(BGE_QUERY_PREFIX, "with prefix"), ("", "without prefix")]:
    r = evaluate_model(bge, eval_queries, documents, query_prefix=prefix, doc_vectors=doc_embeddings[ALT_MODEL_NAME])
    print(f"BGE {label:<15} hit@1 = {r['hit@1']}/15   hit@3 = {r['hit@3']}/15")

**Expected:** the same hit@1 either way on this small set. Always read the model card; on other data it can matter.

## 5. Never mix models
Both models output 384 numbers, but in **different spaces**. Embed the same sentence with each and compare:

In [ ]:
mini = models[MODEL_NAME]
for sentence in ["My customer was charged twice.", "Our Kafka consumer stopped processing messages.",
                 "How do I roll back a failed deployment?"]:
    a = mini.encode([sentence], normalize_embeddings=True)[0]
    b = bge.encode([sentence], normalize_embeddings=True)[0]
    print(f"{a @ b:.3f}  {sentence}")

**Expected:** only ≈ **0.32–0.37**, for the *same* sentence. Cross-model scores are meaningless.

**So:** queries and documents must use the same model. Switching models means **re-embedding every document**.

## 🏠 E3.1: Add your own evaluation query
Write a question an Acme developer might ask, pick the KB document that should answer it (see the table in section 3), then re-run the evaluation.

In [ ]:
# 🏠 E3.1: a question an Acme developer might ask, and the document that should answer it
my_eval_query = {"query": "", "expected_document_id": ""}   # TODO

<details><summary>💡 Hint 1</summary>

Look at `pd.DataFrame(documents)[["document_id", "title"]]` to pick an expected document.

</details>

<details><summary>💡 Hint 2</summary>

Phrase it the way a developer would, not the way the doc does.

</details>

In [ ]:
if not my_eval_query["query"] or not my_eval_query["expected_document_id"]:
    print("E3.1 not finished yet: fill in my_eval_query above.")
else:
    for name, model in models.items():
        r = evaluate_model(model, [my_eval_query], documents, query_prefix=query_prefixes[name],
                           doc_vectors=doc_embeddings[name])
        print(f"{name:<42} rank of {my_eval_query['expected_document_id']}: {r['ranks'][0]}")

## Summary
- Compare models on **your** queries: hit@k plus the failures.
- Weigh quality against size, speed, input length and licence. Bigger ≠ automatically better.
- Never mix vectors from different models.
- We'll keep **MiniLM** for Acme.

➡️ **Notebook 4:** Acme's real runbooks are long. We'll split them into chunks and store them in a vector database.

---
## Solutions
**E3.1:** e.g. `{"query": "Customers get a 'too many requests' error from our API", "expected_document_id": "KB-010"}`: both models rank KB-010 **1st**. Try harder, vaguer questions to find disagreements.